# 01 - Data Quality

Load market data, enforce **point-in-time (PIT)** discipline on a small fundamentals example, and scan illustrative features for **look-ahead leakage**. The examples cover two checks; other backtest failures and a broader audit of the loaded prices remain outside their scope.

> **Data input:** Set exactly one of `QUANTCORTEX_PRICES_CSV` or `QUANTCORTEX_LIVE_YFINANCE=1`. No market data or executed outputs are committed.


In [ ]:
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

logging.getLogger("hmmlearn").setLevel(logging.ERROR)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import matplotlib

try:
    shell = get_ipython()
except NameError:
    shell = None
if shell is None:
    matplotlib.use("Agg")
else:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

# Put the repository root on the path when running from research/.
for candidate in ("..", "."):
    absolute = os.path.abspath(candidate)
    if absolute not in sys.path:
        sys.path.insert(0, absolute)

from quantcortex.backtest.metrics.plotting import (
    INK, NEGATIVE_RED, REFERENCE_BLUE, apply_plot_style, style_axis,
)
from quantcortex.data.local_csv import load_price_matrix

apply_plot_style("notebook")

PRICE_CSV = os.environ.get("QUANTCORTEX_PRICES_CSV")
LIVE_YFINANCE = os.environ.get("QUANTCORTEX_LIVE_YFINANCE") == "1"

if (PRICE_CSV is not None) == LIVE_YFINANCE:
    raise RuntimeError(
        "Set exactly one of QUANTCORTEX_PRICES_CSV=/path/to/prices.csv "
        "or QUANTCORTEX_LIVE_YFINANCE=1."
    )


def load_prices(symbols, start="2018-01-01"):
    """Load adjusted closes from the explicitly selected real-data source."""
    if PRICE_CSV is not None:
        return load_price_matrix(PRICE_CSV, symbols=list(symbols), start=start)

    print(
        "Using live Yahoo Finance data through yfinance; review the provider "
        "terms and confirm that your use is permitted."
    )
    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    prices = YFinanceProvider().get_prices(list(symbols), start=start)
    if prices is None or prices.empty:
        raise RuntimeError("yfinance returned no prices")
    prices = prices.dropna(how="all").ffill(limit=5).dropna()
    if prices.shape[0] <= 120:
        raise RuntimeError("insufficient complete price history from yfinance")
    return prices


selected_source = (
    f"local CSV {Path(PRICE_CSV).expanduser().resolve()}"
    if PRICE_CSV is not None
    else "explicit live yfinance download"
)
print(f"quantcortex research environment ready; source: {selected_source}")


## Load prices
This notebook uses only the explicitly selected real-data source.


In [ ]:
symbols = ["AAPL", "MSFT", "AMZN", "NVDA", "JPM", "XOM"]
prices = load_prices(symbols)
returns = prices.pct_change(fill_method=None).dropna()
print("prices:", prices.shape)
prices.tail()

## Point-in-time enforcement
Fundamental availability follows **announcement_date**, not period_end. With the default date-only policy, `PITEnforcer` requires the recorded feature date to be strictly later than the recorded announcement date. The small fixture illustrates this date comparison. Actual release times and the stated EPS values remain unverified.

In [ ]:
from quantcortex.data.processors.pit_enforcer import PITEnforcer, PITViolationError

fund = pd.DataFrame({
    "symbol": ["AAPL", "AAPL", "MSFT"],
    "period_end": pd.to_datetime(["2023-03-31", "2023-06-30", "2023-03-31"]),
    "announcement_date": pd.to_datetime(["2023-05-04", "2023-08-03", "2023-04-25"]),
    "feature_date": pd.to_datetime(["2023-05-05", "2023-08-04", "2023-04-26"]),
    "field": ["eps", "eps", "eps"],
    "value": [1.52, 1.26, 2.45],
})
clean = PITEnforcer().enforce(fund)
print("PIT enforcement passed:", len(clean), "rows clean")

In [ ]:
# A leaking row: feature_date BEFORE the announcement -> must raise.
bad = fund.copy()
bad.loc[0, "feature_date"] = pd.Timestamp("2023-04-01")  # before the 2023-05-04 announcement
try:
    PITEnforcer().enforce(bad)
    print("ERROR: violation not caught!")
except PITViolationError as e:
    print("Correctly raised PITViolationError:", str(e)[:120])

## Look-ahead leakage scan
This example checks whether `LookaheadDetector` flags `close.shift(-1)` and reports its assessment of a moving average based on past closes. The correlation heuristic can miss leakage or flag a causal feature; it does not certify a feature pipeline.

In [ ]:
from quantcortex.data.processors.lookahead_detector import LookaheadDetector

close = prices.iloc[:, 0]
feat = pd.DataFrame(index=close.index)
feat["causal_ma"] = close.rolling(10).mean().shift(1)   # only past data
feat["causal_ret"] = close.pct_change(fill_method=None).shift(1)
feat["LEAKY_future"] = close.shift(-1)                  # tomorrow used today

report = LookaheadDetector().scan(feat, reference=close)
print(report.summary())
assert "LEAKY_future" in report.flagged_columns
if "causal_ma" in report.flagged_columns:
    print("The causal moving average was flagged. A correlation flag is not proof of leakage.")
print("\nFlagged:", report.flagged_columns)

## Data-quality visualization

The missing-data fraction below describes the cleaned matrix. The loader has already forward-filled up to five missing rows and dropped remaining incomplete rows, so this fraction does not measure missingness in the original input.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
(prices / prices.iloc[0]).plot(ax=ax[0], lw=1.2)
line_styles = ("-", "--", "-.", ":", (0, (5, 2)), (0, (3, 1, 1, 1)))
for line, line_style in zip(ax[0].lines, line_styles):
    line.set_linestyle(line_style)
ax[0].set_title("Normalized prices")
ax[0].set_ylabel("Growth of $1")
ax[0].set_xlabel("Date")
style_axis(ax[0], grid="y")
returns.iloc[:, 0].mul(100).hist(
    bins=60, ax=ax[1], color=REFERENCE_BLUE, edgecolor="white",
)
ax[1].axvline(0.0, color=INK, lw=0.8)
ax[1].set_title(f"{symbols[0]} daily return distribution")
ax[1].set_xlabel("Return (%)")
ax[1].set_ylabel("Sessions")
style_axis(ax[1], grid="y")
plt.tight_layout()
plt.show()
miss = prices.isna().mean().mean()
print(f"missing data fraction: {miss:.4%}")
print("data quality checks complete.")